Criar uma tabela de produtos unindo categoria, marca e quantidade total de estoque de todas as lojas.

In [0]:
bronze_path = '/Volumes/bikestore/bronze/bronze_volumes'
silver_path = '/Volumes/bikestore/silver/silver_volumes'

In [0]:
bronze_tables = {
    'tmp_bronze_brands' : f'{bronze_path}/brands/',
    'tmp_bronze_categories' : f'{bronze_path}/categories/',
    'tmp_bronze_customers' : f'{bronze_path}/customers/',
    'tmp_bronze_order_items' : f'{bronze_path}/order_items',
    'tmp_bronze_orders' : f'{bronze_path}/orders/',
    'tmp_bronze_products' : f'{bronze_path}/products/',
    'tmp_bronze_staffs' : f'{bronze_path}/staffs/',
    'tmp_bronze_stocks' : f'{bronze_path}/stocks/',
    'tmp_bronze_stores' : f'{bronze_path}/stores/'
}

for view_name, path in bronze_tables.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name))

In [0]:
%sql
with stock as (
  select
    product_id,
    sum(quantity) as total_stock
  from
    tmp_bronze_stocks
  group by
    product_id
)

select
  product.product_id,
  product.product_name,
  brand.brand_name,
  category.category_name,
  product.model_year,
  product.list_price,
  s.total_stock
from
  tmp_bronze_products as product
    left join tmp_bronze_categories as category
      on product.category_id = category.category_id
    left join tmp_bronze_brands as brand
      on product.brand_id = brand.brand_id
    left join stock as s
      on product.product_id = s.product_id

In [0]:
# Transformando em formato DELTA Silver
df_product_silver = spark.sql("""
with stock as (
  select
    product_id,
    sum(quantity) as total_stock
  from
    tmp_bronze_stocks
  group by
    product_id
)

select
  product.product_id,
  product.product_name,
  brand.brand_name,
  category.category_name,
  product.model_year,
  product.list_price,
  s.total_stock
from
  tmp_bronze_products as product
    left join tmp_bronze_categories as category
      on product.category_id = category.category_id
    left join tmp_bronze_brands as brand
      on product.brand_id = brand.brand_id
    left join stock as s
      on product.product_id = s.product_id  
                               """)


In [0]:
display(df_product_silver)

In [0]:
# Salvando em DELTA na camada Silver
df_product_silver.write.mode('overwrite').format('delta').option('mergeShchema', 'true').save(f'{silver_path}/product')

In [0]:
%sql
-- Criando uma tabela física

CREATE TABLE bikestore.silver.silver_products
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/bikestore/silver/silver_volumes/orders/`;